# binbase Database Info

This notebook connects to the **binbase** TimescaleDB instance and queries database metadata:
available symbols, database size, table sizes, hypertable details, row counts, and sample data.

> **Requires** a running binbase server at `192.168.100.221:5432`.
> Password can be passed directly or set via the `BINBASE_PASSWORD` environment variable.

In [1]:
import pandas as pd

from binpan.storage.binbase import connect, get_symbols, close
from binpan.storage.postgresql_database import get_db_size, get_table_sizes, get_hypertable_info

## Connect to binbase

In [2]:
conn, cur = connect()  # password from BINBASE_PASSWORD env var or secret.py

## Available Symbols

In [3]:
symbols = get_symbols(cur)
print(f"{len(symbols)} symbols available:")
symbols

10 symbols available:


['BNBUSDC',
 'BTCEUR',
 'BTCUSDC',
 'DOGEUSDC',
 'ETHEUR',
 'ETHUSDC',
 'SOLEUR',
 'SOLUSDC',
 'SUIUSDC',
 'XRPUSDC']

## Database Size

In [4]:
print(f"Database size: {get_db_size(cur)}")

Database size: 5662 MB


## Table Sizes (public schema)

In [5]:
get_table_sizes(cur, only_public_schema=True)

,schemaname,tablename,table_size_kb,indexes_size_kb,total_size_kb
0,public,trades,8.0,32.0,40.0
1,public,agg_trades,8.0,24.0,32.0
2,public,klines,8.0,24.0,32.0
3,public,book_tickers,8.0,16.0,24.0
4,public,tickers,8.0,16.0,24.0
5,public,orderbook_snapshots,8.0,16.0,24.0


## Hypertable Details

In [6]:
get_hypertable_info(cur)

,hypertable_schema,hypertable_name,owner,num_dimensions,num_chunks,compression_enabled,tablespaces,primary_dimension,primary_dimension_type
0,public,trades,binbase,1,3,True,None,time,timestamp with time zone
2,public,orderbook_snapshots,binbase,1,2,True,None,time,timestamp with time zone
1,public,agg_trades,binbase,1,0,True,None,time,timestamp with time zone
3,public,klines,binbase,1,0,True,None,time,timestamp with time zone
4,public,book_tickers,binbase,1,0,True,None,time,timestamp with time zone
5,public,tickers,binbase,1,0,True,None,time,timestamp with time zone


## Row Counts

Direct SQL queries to count rows in the main hypertables and views.

In [7]:
tables_to_count = ["trades", "agg_trades", "orderbook_snapshots"]
counts = {}
for table in tables_to_count:
    try:
        cur.execute(f"SELECT count(*) FROM {table};")
        counts[table] = cur.fetchone()[0]
    except Exception as e:
        counts[table] = f"error: {e}"
        conn.rollback()

pd.Series(counts, name="row_count")

trades                 20789765
agg_trades                    0
orderbook_snapshots      559940
Name: row_count, dtype: int64

## Sample Data

A few rows from the `trades` table and the `klines_1h` continuous aggregate view.

In [8]:
cur.execute("SELECT * FROM trades ORDER BY time DESC LIMIT 5;")
cols = [desc[0] for desc in cur.description]
pd.DataFrame(cur.fetchall(), columns=cols)

,time,symbol,market,trade_id,price,quantity,quote_qty,buyer_maker
0,2026-03-14 22:17:25.562000+00:00,BTCUSDC,spot,348606910,70751.00000000,0.00027000,19.10277000,False
1,2026-03-14 22:17:25.383000+00:00,BTCUSDC,spot,348606900,70748.13000000,0.00008000,5.65985040,False
2,2026-03-14 22:17:25.383000+00:00,BTCUSDC,spot,348606901,70748.13000000,0.00008000,5.65985040,False
3,2026-03-14 22:17:25.383000+00:00,BTCUSDC,spot,348606902,70748.13000000,0.00008000,5.65985040,False
4,2026-03-14 22:17:25.383000+00:00,BTCUSDC,spot,348606903,70748.13000000,0.00008000,5.65985040,False


In [9]:
cur.execute("SELECT * FROM klines_1h ORDER BY bucket DESC LIMIT 5;")
cols = [desc[0] for desc in cur.description]
pd.DataFrame(cur.fetchall(), columns=cols)

,bucket,symbol,market,open,high,low,close,volume,quote_volume,trades,taker_buy_volume,taker_buy_quote_volume
0,2026-03-14 22:00:00+00:00,BNBUSDC,spot,654.95000000,655.63000000,654.56000000,654.66000000,107.45200000,70399.58655000,296,74.41600000,48753.68536000
1,2026-03-14 22:00:00+00:00,BTCEUR,spot,62082.92000000,62091.30000000,62000.39000000,62020.60000000,0.83713000,51943.30524210,182,0.46049000,28572.43699060
2,2026-03-14 22:00:00+00:00,BTCUSDC,spot,70820.04000000,70821.58000000,70724.65000000,70751.00000000,9.69188000,685963.00275800,2364,4.45233000,315086.92629800
3,2026-03-14 22:00:00+00:00,DOGEUSDC,spot,0.09508000,0.09516000,0.09494000,0.09497000,122778.00000000,11671.31744000,142,64549.00000000,6135.63631000
4,2026-03-14 22:00:00+00:00,ETHEUR,spot,1829.10000000,1831.01000000,1827.99000000,1828.62000000,16.52000000,30223.70975700,94,12.93530000,23665.26187300


## Close Connection

In [10]:
close(conn)
print("Connection closed.")

Connection closed.
